# 10 — RQ1 final build + held-out test analysis

Trains the three frozen classifiers (RoBERTa for 1A; DeBERTa-v3 for 1B and 2) on `train_fit`, selecting checkpoints on `validation` — no test access. Then the analysis chain loads the saved test-set probabilities, applies the fixed decision rules (1A ≥ 0.46; 1B ≥ 0.59 **and** 1A positive; 2 ≥ 0.45), and reports positive-class metrics with bootstrap 95% CIs.

**Requires a GPU** for the training cell. Paths are repo-relative: data in `data/processed/`, trained models in `final_models/`, results in `outputs/rq1/`.


In [ ]:
# ============================================================
# FINAL RQ1 MODEL BUILD
# ============================================================
#
# Trains and saves three independent binary classifiers.
#
# DEVELOPMENT DATA:
#   data/processed/train_fit.csv
#   data/processed/validation.csv
#
# FINAL PREDICTION RULES:
#   1A = P(1A) >= 0.46
#   1B = (P(1B) >= 0.59) AND (P(1A) >= 0.46)
#   2  = P(2) >= 0.45
#
# The 1B constraint is applied after model inference.
# All reported final 1B metrics use constrained predictions.
#
# Training and checkpoint selection remain independent
# for each classifier. The constraint was introduced after
# the original independent-model evaluation.
#
# No test or RQ2 data are used in this training cell.
# ============================================================

In [ ]:
# ============================================================
# RQ1 MODEL BUILD — NO TEST ACCESS
# ============================================================
#
# Builds and saves the three frozen final classifiers.
#
# DATA USED:
#   data/processed/train_fit.csv
#   data/processed/validation.csv
#
# DOES NOT LOAD:
#   test.csv
#
# FROZEN MODELS
# --------------
# 1A -> RoBERTa-base
#       LR 3e-5 | WD 0.00 | warmup 0 | threshold 0.46
#
# 1B -> DeBERTa-v3-base
#       LR 3e-5 | WD 0.01 | warmup 0 | threshold 0.59
#
# 2  -> DeBERTa-v3-base
#       LR 2e-5 | WD 0.00 | warmup 0 | threshold 0.45
#
# Training:
# train_fit -> fitting
# validation -> checkpoint selection
#
# No tuning is performed here.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import gc
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from datasets import Dataset

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train_fit.csv"
VAL_PATH = DATA_DIR / "validation.csv"

FINAL_MODEL_DIR = PROJECT_ROOT / "final_models"
TEMP_RUN_DIR = PROJECT_ROOT / "final_training_runs"

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEMP_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD DEVELOPMENT DATA ONLY
# ============================================================

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        "train_fit.csv not found in data/processed/"
    )

if not VAL_PATH.exists():
    raise FileNotFoundError(
        "validation.csv not found in data/processed/"
    )

train_fit = pd.read_csv(TRAIN_PATH)
val = pd.read_csv(VAL_PATH)

print("=" * 72)
print("FINAL MODEL BUILD — DEVELOPMENT DATA ONLY")
print("=" * 72)

print(f"train_fit rows: {len(train_fit)}")
print(f"validation rows: {len(val)}")


# ============================================================
# 3. SAFETY CHECKS
# ============================================================

EXPECTED_TRAIN_ROWS = 1215
EXPECTED_VAL_ROWS = 405

EXPECTED_POSITIVES = {

    "label_1A": {
        "train": 377,
        "val": 126
    },

    "label_1B": {
        "train": 118,
        "val": 40
    },

    "label_2": {
        "train": 98,
        "val": 33
    }
}


if len(train_fit) != EXPECTED_TRAIN_ROWS:
    raise ValueError(
        f"Unexpected train_fit size: {len(train_fit)}"
    )

if len(val) != EXPECTED_VAL_ROWS:
    raise ValueError(
        f"Unexpected validation size: {len(val)}"
    )


required_columns = {
    "text_clean",
    "label_1A",
    "label_1B",
    "label_2"
}


for name, df in [
    ("train_fit", train_fit),
    ("validation", val)
]:

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )


for label, expected in EXPECTED_POSITIVES.items():

    train_pos = int(
        train_fit[label].sum()
    )

    val_pos = int(
        val[label].sum()
    )

    if train_pos != expected["train"]:
        raise ValueError(
            f"{label}: unexpected train "
            f"positive count {train_pos}"
        )

    if val_pos != expected["val"]:
        raise ValueError(
            f"{label}: unexpected validation "
            f"positive count {val_pos}"
        )


print("\nSplit checks passed.")

for label in [
    "label_1A",
    "label_1B",
    "label_2"
]:

    print(
        f"{label}: "
        f"train positives={int(train_fit[label].sum())}, "
        f"validation positives={int(val[label].sum())}"
    )


# ============================================================
# 4. FROZEN FINAL CONFIGURATIONS
# ============================================================

CONFIGS = {

    "label_1A": {

        "architecture": "RoBERTa",

        "model_name":
            "roberta-base",

        "learning_rate":
            3e-5,

        "weight_decay":
            0.00,

        "warmup_steps":
            0,

        "threshold":
            0.46,

        "train_batch":
            16,

        "eval_batch":
            32,

        "grad_accum":
            1,

        "force_fp32":
            False
    },


    "label_1B": {

        "architecture": "DeBERTa",

        "model_name":
            "microsoft/deberta-v3-base",

        "learning_rate":
            3e-5,

        "weight_decay":
            0.01,

        "warmup_steps":
            0,

        "threshold":
            0.59,

        "train_batch":
            8,

        "eval_batch":
            16,

        "grad_accum":
            2,

        "force_fp32":
            True
    },


    "label_2": {

        "architecture": "DeBERTa",

        "model_name":
            "microsoft/deberta-v3-base",

        "learning_rate":
            2e-5,

        "weight_decay":
            0.00,

        "warmup_steps":
            0,

        "threshold":
            0.45,

        "train_batch":
            8,

        "eval_batch":
            16,

        "grad_accum":
            2,

        "force_fp32":
            True
    }
}


SEED = 42
NUM_EPOCHS = 3
MAX_LENGTH = 256

set_seed(SEED)


# ============================================================
# 5. HELPERS
# ============================================================

def clear_memory():

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def make_dataset(
    df,
    label,
    tokenizer
):

    ds = Dataset.from_pandas(

        df[
            [
                "text_clean",
                label
            ]
        ].rename(
            columns={
                label: "labels"
            }
        ),

        preserve_index=False
    )

    ds = ds.map(

        lambda batch: tokenizer(

            batch["text_clean"],

            truncation=True,

            max_length=MAX_LENGTH
        ),

        batched=True
    )

    return ds


def compute_metrics(
    eval_pred
):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=1
    )

    return {

        "precision":
            precision_score(
                labels,
                preds,
                zero_division=0
            ),

        "recall":
            recall_score(
                labels,
                preds,
                zero_division=0
            ),

        "f1":
            f1_score(
                labels,
                preds,
                zero_division=0
            )
    }


# ============================================================
# 6. PROVENANCE STORAGE
# ============================================================

build_rows = []


# ============================================================
# 7. BUILD EACH FINAL MODEL
# ============================================================

for label, cfg in CONFIGS.items():

    print("\n")
    print("=" * 72)

    print(
        f"BUILDING FINAL {label} MODEL"
    )

    print("=" * 72)

    print(
        f"Architecture: {cfg['architecture']}"
    )

    print(
        f"Model: {cfg['model_name']}"
    )

    print(
        f"Learning rate: "
        f"{cfg['learning_rate']:.0e}"
    )

    print(
        f"Weight decay: "
        f"{cfg['weight_decay']}"
    )

    print(
        f"Warmup steps: "
        f"{cfg['warmup_steps']}"
    )

    print(
        f"Frozen threshold: "
        f"{cfg['threshold']}"
    )


    clear_memory()
    set_seed(SEED)


    # ========================================================
    # TOKENIZER
    # ========================================================

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            cfg["model_name"]
        )
    )


    data_collator = (
        DataCollatorWithPadding(
            tokenizer=tokenizer
        )
    )


    # ========================================================
    # DATASETS
    # ========================================================

    train_ds = make_dataset(
        train_fit,
        label,
        tokenizer
    )

    val_ds = make_dataset(
        val,
        label,
        tokenizer
    )


    # ========================================================
    # CLASS WEIGHTS
    # ========================================================

    weights = compute_class_weight(

        class_weight="balanced",

        classes=np.array(
            [0, 1]
        ),

        y=train_fit[
            label
        ].values
    )


    class_weights = torch.tensor(
        weights,
        dtype=torch.float32
    )


    print(
        "\nClass weights:",
        class_weights.tolist()
    )


    # ========================================================
    # WEIGHTED TRAINER
    # ========================================================

    class WeightedTrainer(Trainer):

        def compute_loss(
            self,
            model,
            inputs,
            return_outputs=False,
            **kwargs
        ):

            labels = inputs.pop(
                "labels"
            )

            outputs = model(
                **inputs
            )

            logits = (
                outputs
                .logits
                .float()
            )

            weights_device = (
                class_weights
                .to(
                    device=
                        logits.device,

                    dtype=
                        torch.float32
                )
            )

            loss = nn.CrossEntropyLoss(
                weight=
                    weights_device
            )(
                logits,
                labels
            )

            return (
                (loss, outputs)
                if return_outputs
                else loss
            )


    # ========================================================
    # MODEL
    # ========================================================

    model_kwargs = {
        "num_labels": 2
    }


    # Explicit FP32 for DeBERTa because the earlier
    # mixed-precision attempt produced NaNs.
    if cfg["force_fp32"]:

        model_kwargs[
            "dtype"
        ] = torch.float32


    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            cfg["model_name"],
            **model_kwargs
        )
    )


    # ========================================================
    # TEMP RUN DIRECTORY
    # ========================================================

    run_path = (
        TEMP_RUN_DIR
        / label
    )


    if run_path.exists():

        shutil.rmtree(
            run_path
        )


    # ========================================================
    # TRAINING ARGUMENTS
    #
    # Frozen development procedure:
    #
    # - 3 epochs
    # - evaluation every 25 steps
    # - save every 25 steps
    # - validation F1 selects checkpoint
    # - NO hyperparameter tuning
    # - NO threshold tuning
    #
    # ========================================================

    training_args = TrainingArguments(

        output_dir=
            str(run_path),

        num_train_epochs=
            NUM_EPOCHS,

        per_device_train_batch_size=
            cfg["train_batch"],

        per_device_eval_batch_size=
            cfg["eval_batch"],

        gradient_accumulation_steps=
            cfg["grad_accum"],

        learning_rate=
            cfg["learning_rate"],

        weight_decay=
            cfg["weight_decay"],

        # All three frozen winners used zero warmup.
        # Using warmup_steps because current Transformers
        # version does not accept warmup_ratio.
        warmup_steps=
            cfg["warmup_steps"],

        eval_strategy=
            "steps",

        save_strategy=
            "steps",

        eval_steps=
            25,

        save_steps=
            25,

        load_best_model_at_end=
            True,

        metric_for_best_model=
            "f1",

        greater_is_better=
            True,

        save_total_limit=
            1,

        logging_steps=
            25,

        report_to=
            "none",

        seed=
            SEED,

        data_seed=
            SEED,

        fp16=
            False,

        bf16=
            False
    )


    # ========================================================
    # TRAINER
    # ========================================================

    trainer = WeightedTrainer(

        model=model,

        args=
            training_args,

        train_dataset=
            train_ds,

        eval_dataset=
            val_ds,

        data_collator=
            data_collator,

        compute_metrics=
            compute_metrics
    )


    # ========================================================
    # TRAIN
    # ========================================================

    trainer.train()


    # ========================================================
    # RECORD BEST VALIDATION CHECKPOINT
    # ========================================================

    best_validation_f1 = float(
        trainer.state.best_metric
    )


    best_checkpoint = (
        trainer.state
        .best_model_checkpoint
    )


    print("\n")
    print(
        f"Best validation checkpoint F1: "
        f"{best_validation_f1:.3f}"
    )

    print(
        f"Selected checkpoint: "
        f"{best_checkpoint}"
    )


    # ========================================================
    # SAVE EXACT BEST MODEL
    # ========================================================

    folder_name = (
        f"{label}_"
        f"{cfg['architecture'].lower()}"
    )


    final_path = (
        FINAL_MODEL_DIR
        / folder_name
    )


    if final_path.exists():

        shutil.rmtree(
            final_path
        )


    # trainer currently holds the best validation checkpoint
    # because load_best_model_at_end=True.
    trainer.save_model(
        str(final_path)
    )


    tokenizer.save_pretrained(
        str(final_path)
    )


    # ========================================================
    # SAVE MODEL METADATA
    # ========================================================

    metadata = {

        "label":
            label,

        "architecture":
            cfg["architecture"],

        "model_name":
            cfg["model_name"],

        "seed":
            SEED,

        "epochs":
            NUM_EPOCHS,

        "learning_rate":
            cfg["learning_rate"],

        "weight_decay":
            cfg["weight_decay"],

        "warmup_steps":
            cfg["warmup_steps"],

        "frozen_threshold":
            cfg["threshold"],

        "train_batch":
            cfg["train_batch"],

        "eval_batch":
            cfg["eval_batch"],

        "gradient_accumulation":
            cfg["grad_accum"],

        "max_length":
            MAX_LENGTH,

        "best_validation_checkpoint_f1":
            best_validation_f1,

        "selected_checkpoint":
            best_checkpoint
    }


    with open(
        final_path
        / "final_model_metadata.json",
        "w"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2
        )


    build_rows.append(
        metadata
    )


    print(
        f"\nSaved final model to:\n"
        f"{final_path}"
    )


    # ========================================================
    # CLEAN TEMP CHECKPOINTS / MEMORY
    # ========================================================

    del trainer
    del model
    del tokenizer
    del train_ds
    del val_ds

    shutil.rmtree(
        run_path,
        ignore_errors=True
    )

    clear_memory()


# ============================================================
# 8. SAVE MASTER CONFIG TABLE
# ============================================================

build_df = pd.DataFrame(
    build_rows
)


CONFIG_OUTPUT_PATH = (
    DATA_DIR
    / "rq1_final_model_build_config.csv"
)


build_df.to_csv(
    CONFIG_OUTPUT_PATH,
    index=False
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 72)
print("FINAL MODEL BUILD COMPLETE")
print("=" * 72)


print(
    build_df[
        [
            "label",
            "architecture",
            "learning_rate",
            "weight_decay",
            "warmup_steps",
            "frozen_threshold",
            "best_validation_checkpoint_f1"
        ]
    ]
    .round(3)
    .to_string(
        index=False
    )
)


print("\nSaved model folders:")

for label, cfg in CONFIGS.items():

    folder_name = (
        f"{label}_"
        f"{cfg['architecture'].lower()}"
    )

    path = (
        FINAL_MODEL_DIR
        / folder_name
    )

    print(
        f"  {path}"
    )


print(
    "\nSaved configuration table:"
)

print(
    CONFIG_OUTPUT_PATH
)


print("\nIMPORTANT:")

print(
    "No test data were loaded or evaluated."
)

print(
    "No RQ2 data were loaded or scored."
)

print("\nDONE.")

In [ ]:
# ============================================================
# RQ1_ANALYSIS - SETUP AND VERIFY SAVED TEST PROBABILITIES
#
# Uses probabilities from the existing frozen classifiers.
#
# Final prediction rules:
#   pred_1A = prob_1A >= 0.46
#   pred_1B = (prob_1B >= 0.59) AND (pred_1A == 1)
#   pred_2  = prob_2 >= 0.45
#
# The 1B constraint was introduced after the original evaluation.
# Original input files are preserved.
# ============================================================

from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


# ============================================================
# 1. PATHS
# ============================================================

RESULTS_DIR = PROJECT_ROOT / "outputs" / "rq1"

INPUT_PATH = (
    RESULTS_DIR / "rq1_final_test_predictions.csv"
)

OUTPUT_DIR = (
    RESULTS_DIR / "RQ1_ANALYSIS"
)

print("Input file:", INPUT_PATH)
print("File exists:", INPUT_PATH.is_file())

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Required file not found:\n{INPUT_PATH}\n\n"
        "The expected filename is "
        "rq1_final_test_predictions.csv."
    )


# ============================================================
# 2. FIXED SETTINGS
# ============================================================

THRESHOLDS = {
    "1A": 0.46,
    "1B": 0.59,
    "2": 0.45,
}

SEED = 42
N_BOOTSTRAPS = 10_000

LABEL_COLUMNS = [
    "label_1A",
    "label_1B",
    "label_2",
]

PROBABILITY_COLUMNS = [
    "prob_1A",
    "prob_1B",
    "prob_2",
]


# ============================================================
# 3. LOAD ORIGINAL TEST OUTPUT
# ============================================================

source = pd.read_csv(
    INPUT_PATH,
    encoding="utf-8-sig",
    dtype={"comment_id": "string"},
)

required_columns = {
    "text_clean",
    *LABEL_COLUMNS,
    *PROBABILITY_COLUMNS,
}

missing_columns = required_columns - set(source.columns)

if missing_columns:
    raise ValueError(
        "Missing required columns:\n"
        f"{sorted(missing_columns)}"
    )

if len(source) != 406:
    raise ValueError(
        f"Expected 406 held-out test comments; "
        f"found {len(source)}."
    )


# ============================================================
# 4. VERIFY HUMAN LABELS
# ============================================================

for column in LABEL_COLUMNS:
    values = pd.to_numeric(
        source[column],
        errors="raise",
    )

    if not values.isin([0, 1]).all():
        raise ValueError(
            f"{column} contains missing or non-binary labels."
        )

    source[column] = values.astype(int)


expected_positives = {
    "label_1A": 126,
    "label_1B": 39,
    "label_2": 32,
}

for column, expected in expected_positives.items():
    observed = int(source[column].sum())

    if observed != expected:
        raise ValueError(
            f"{column}: expected {expected} positive labels; "
            f"found {observed}."
        )


human_hierarchy_violations = int(
    (
        (source["label_1B"] == 1)
        & (source["label_1A"] == 0)
    ).sum()
)

if human_hierarchy_violations:
    raise ValueError(
        "Human labels contain "
        f"{human_hierarchy_violations} cases where "
        "label_1B=1 but label_1A=0."
    )


# ============================================================
# 5. VERIFY SAVED PROBABILITIES
# ============================================================

for column in PROBABILITY_COLUMNS:
    values = pd.to_numeric(
        source[column],
        errors="raise",
    )

    if not np.isfinite(
        values.to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"{column} contains missing or non-finite scores."
        )

    if not values.between(0, 1).all():
        raise ValueError(
            f"{column} contains scores outside [0, 1]."
        )

    source[column] = values.astype(float)


# ============================================================
# 6. VERIFY COMMENT TEXT AND IDS
# ============================================================

if source["text_clean"].isna().any():
    raise ValueError("Missing comment text.")

if source["text_clean"].astype(str).str.strip().eq("").any():
    raise ValueError("Blank comment text.")

if source["text_clean"].duplicated().any():
    raise ValueError(
        "Duplicate exact texts found in the test file."
    )

if "comment_id" in source.columns:
    ids = source["comment_id"]

    if (
        ids.isna().any()
        or ids.str.strip().eq("").any()
        or not ids.is_unique
    ):
        raise ValueError(
            "Missing, blank or duplicate comment IDs."
        )


# ============================================================
# 7. PREPARE EVALUATION DATA
#
# Existing prediction columns are not copied into this table.
# The next cell creates final predictions using the fixed rules.
# ============================================================

keep_columns = (
    (["comment_id"] if "comment_id" in source.columns else [])
    + ["text_clean"]
    + LABEL_COLUMNS
    + PROBABILITY_COLUMNS
)

evaluation = source[keep_columns].copy()

SOURCE_SHA256 = hashlib.sha256(
    INPUT_PATH.read_bytes()
).hexdigest()

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 8. CONFIRMATION
# ============================================================

print("\n" + "=" * 72)
print("RQ1_ANALYSIS — INPUT VERIFIED")
print("=" * 72)

print(f"\nHeld-out test comments: {len(evaluation)}")

print("\nHuman positive counts:")
print(evaluation[LABEL_COLUMNS].sum())

print(
    "\nHuman hierarchy violations:",
    human_hierarchy_violations,
)

print("\nFixed thresholds:")
for construct, threshold in THRESHOLDS.items():
    print(f"{construct}: {threshold:.2f}")

print("\nFinal prediction rule for 1B:")
print(
    "pred_1B = 1 only when prob_1B >= 0.59 "
    "and pred_1A = 1."
)

print("\nOutput folder:")
print(OUTPUT_DIR)



In [ ]:
# ============================================================
# RQ1_ANALYSIS - APPLY AND SAVE FINAL PREDICTION RULES
#
# All 406 comments remain in the evaluation.
# pred_1B is the constrained prediction.
# ============================================================

import json
from datetime import datetime, timezone


# ------------------------------------------------------------
# 1. CHECK THAT CELL 1 HAS RUN
# ------------------------------------------------------------

required_objects = [
    "evaluation",
    "THRESHOLDS",
    "OUTPUT_DIR",
    "INPUT_PATH",
    "SOURCE_SHA256",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run Cell 1 first. Missing objects: "
        + ", ".join(missing_objects)
    )


# ------------------------------------------------------------
# 2. APPLY FIXED PREDICTION RULES
# ------------------------------------------------------------

evaluation = evaluation.copy()

evaluation["pred_1A"] = (
    evaluation["prob_1A"] >= THRESHOLDS["1A"]
).astype(int)

evaluation["pred_1B"] = (
    (evaluation["prob_1B"] >= THRESHOLDS["1B"])
    & (evaluation["pred_1A"] == 1)
).astype(int)

evaluation["pred_2"] = (
    evaluation["prob_2"] >= THRESHOLDS["2"]
).astype(int)


# ------------------------------------------------------------
# 3. VERIFY FINAL PREDICTIONS
# ------------------------------------------------------------

prediction_columns = [
    "pred_1A",
    "pred_1B",
    "pred_2",
]

assert len(evaluation) == 406

assert evaluation[
    prediction_columns
].isin([0, 1]).all().all()

prediction_hierarchy_violations = int(
    (
        (evaluation["pred_1B"] == 1)
        & (evaluation["pred_1A"] == 0)
    ).sum()
)

assert prediction_hierarchy_violations == 0, (
    "A predicted 1B-positive/1A-negative case was found."
)


# ------------------------------------------------------------
# 4. SAVE FINAL PREDICTIONS
# ------------------------------------------------------------

PREDICTIONS_PATH = (
    OUTPUT_DIR / "rq1_final_constrained_test_predictions.csv"
)

evaluation.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 5. SAVE ANALYSIS SETTINGS AND PROVENANCE
# ------------------------------------------------------------

METADATA_PATH = (
    OUTPUT_DIR / "rq1_analysis_settings.json"
)

settings = {
    "analysis": "RQ1_ANALYSIS",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(INPUT_PATH),
    "source_sha256": SOURCE_SHA256,
    "n_test": len(evaluation),
    "thresholds": THRESHOLDS,
    "prediction_rules": {
        "pred_1A": "prob_1A >= 0.46",
        "pred_1B": "(prob_1B >= 0.59) AND (pred_1A == 1)",
        "pred_2": "prob_2 >= 0.45",
    },
    "constraint_timing": (
        "Introduced after the original independent-model "
        "test evaluation."
    ),
    "models_retrained": False,
    "thresholds_retuned": False,
    "comments_filtered": False,
    "bootstrap_seed": SEED,
    "n_bootstraps": N_BOOTSTRAPS,
}

with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2)


# ------------------------------------------------------------
# 6. DISPLAY SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame({
    "construct": ["1A", "1B", "2"],
    "human_positive_n": [
        int(evaluation["label_1A"].sum()),
        int(evaluation["label_1B"].sum()),
        int(evaluation["label_2"].sum()),
    ],
    "predicted_positive_n": [
        int(evaluation["pred_1A"].sum()),
        int(evaluation["pred_1B"].sum()),
        int(evaluation["pred_2"].sum()),
    ],
})

print("=" * 72)
print("RQ1_ANALYSIS — FINAL PREDICTIONS CREATED")
print("=" * 72)

print(f"\nTest comments retained: {len(evaluation)}")
print(
    "Predicted hierarchy violations:",
    prediction_hierarchy_violations,
)

print("\n")
print(summary.to_string(index=False))

print("\nSaved:")
print(PREDICTIONS_PATH)
print(METADATA_PATH)


In [ ]:
# ============================================================
# RQ1_ANALYSIS
# CELL 3 — FINAL PERFORMANCE AND BOOTSTRAP INTERVALS
#
# Uses the saved final predictions from Cell 2.
# All 1B results use constrained predictions.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. LOAD FINAL PREDICTIONS AND SETTINGS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rq1" / "RQ1_ANALYSIS"

PREDICTIONS_PATH = (
    OUTPUT_DIR / "rq1_final_constrained_test_predictions.csv"
)

SETTINGS_PATH = (
    OUTPUT_DIR / "rq1_analysis_settings.json"
)

METRICS_PATH = (
    OUTPUT_DIR / "rq1_final_constrained_test_metrics.csv"
)

for path in [PREDICTIONS_PATH, SETTINGS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required file not found:\n{path}\nRun Cell 2 first."
        )

evaluation = pd.read_csv(
    PREDICTIONS_PATH,
    encoding="utf-8-sig",
)

with SETTINGS_PATH.open(encoding="utf-8") as file:
    settings = json.load(file)

SEED = int(settings["bootstrap_seed"])
N_BOOTSTRAPS = int(settings["n_bootstraps"])

LABEL_PAIRS = {
    "1A": ("label_1A", "pred_1A"),
    "1B": ("label_1B", "pred_1B"),
    "2": ("label_2", "pred_2"),
}

assert len(evaluation) == 406

for actual_column, prediction_column in LABEL_PAIRS.values():
    for column in [actual_column, prediction_column]:
        values = pd.to_numeric(
            evaluation[column],
            errors="raise",
        )
        assert values.isin([0, 1]).all(), (
            f"Missing or invalid binary values in {column}."
        )
        evaluation[column] = values.astype(int)

assert not (
    (evaluation["pred_1B"] == 1)
    & (evaluation["pred_1A"] == 0)
).any(), "Predicted hierarchy violation found."


# ------------------------------------------------------------
# 2. METRIC FUNCTIONS
# ------------------------------------------------------------

def confusion_counts(actual, predicted):
    tn = int(((actual == 0) & (predicted == 0)).sum())
    fp = int(((actual == 0) & (predicted == 1)).sum())
    fn = int(((actual == 1) & (predicted == 0)).sum())
    tp = int(((actual == 1) & (predicted == 1)).sum())

    return tn, fp, fn, tp


def performance(actual, predicted):
    tn, fp, fn, tp = confusion_counts(actual, predicted)

    # Same zero-denominator convention as the original evaluation.
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1_denominator = 2 * tp + fp + fn
    f1 = (
        2 * tp / f1_denominator
        if f1_denominator else 0.0
    )

    return np.array(
        [precision, recall, f1],
        dtype=float,
    )


# ------------------------------------------------------------
# 3. PREPARE ARRAYS
# ------------------------------------------------------------

arrays = {
    construct: (
        evaluation[actual_column].to_numpy(),
        evaluation[prediction_column].to_numpy(),
    )
    for construct, (actual_column, prediction_column)
    in LABEL_PAIRS.items()
}

metric_names = ["precision", "recall", "f1"]

bootstrap_results = {
    construct: np.empty((N_BOOTSTRAPS, 3), dtype=float)
    for construct in LABEL_PAIRS
}


# ------------------------------------------------------------
# 4. BOOTSTRAP HELD-OUT COMMENTS
#
# Sample 406 comments with replacement on each iteration.
# Use the same resampled indices for all three constructs.
# Predictions, thresholds and models remain fixed.
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

print(
    f"Running {N_BOOTSTRAPS:,} bootstrap resamples "
    f"over {len(evaluation)} test comments..."
)

for iteration in range(N_BOOTSTRAPS):
    indices = rng.integers(
        0,
        len(evaluation),
        size=len(evaluation),
    )

    for construct, (actual, predicted) in arrays.items():
        bootstrap_results[construct][iteration] = performance(
            actual[indices],
            predicted[indices],
        )


# ------------------------------------------------------------
# 5. BUILD FINAL METRICS TABLE
# ------------------------------------------------------------

rows = []

for construct, (actual, predicted) in arrays.items():
    tn, fp, fn, tp = confusion_counts(actual, predicted)
    point_estimates = performance(actual, predicted)

    row = {
        "construct": construct,
        "prediction_rule": (
            "1B_requires_predicted_1A"
            if construct == "1B"
            else "fixed_threshold"
        ),
        "n_test": len(actual),
        "human_positive_n": int(actual.sum()),
        "predicted_positive_n": int(predicted.sum()),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": (tn + tp) / len(actual),
    }

    for index, metric in enumerate(metric_names):
        low, high = np.percentile(
            bootstrap_results[construct][:, index],
            [2.5, 97.5],
        )

        row[metric] = float(point_estimates[index])
        row[f"{metric}_ci_low"] = float(low)
        row[f"{metric}_ci_high"] = float(high)

    rows.append(row)

metrics = pd.DataFrame(rows)


# ------------------------------------------------------------
# 6. SAVE FULL-PRECISION RESULTS
# ------------------------------------------------------------

metrics.to_csv(
    METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("RQ1_ANALYSIS — FINAL HELD-OUT PERFORMANCE")
print("=" * 72)

for _, row in metrics.iterrows():
    print(f"\nConstruct {row['construct']}")

    for metric in metric_names:
        print(
            f"{metric.capitalize():9s}: "
            f"{row[metric]:.3f} "
            f"[{row[f'{metric}_ci_low']:.3f}, "
            f"{row[f'{metric}_ci_high']:.3f}]"
        )

    print(
        f"TN={int(row['tn'])} | FP={int(row['fp'])} | "
        f"FN={int(row['fn'])} | TP={int(row['tp'])}"
    )

print("\nAll 1B metrics use constrained predictions.")

print("\nSaved:")
print(METRICS_PATH)

In [ ]:
# FIGURES
import numpy as np
import matplotlib.pyplot as plt

# ---- edit these with your numbers ----
constructs = ["1A\ngrievance", "1B\nweaponised", "2\nconspiracy"]
results = {                       # model: [1A, 1B, 2] positive-class F1
    "Lexicon":              [.312, .169, .278],
    "Naïve Bayes":          [.535, .274, .416],
    "Logistic regression":  [.564, .382, .480],
    "RoBERTa-base":         [.661, .524, .655],   # validation F1
    "DeBERTa-v3-base":      [.659, .540, .685],   # validation F1
}
colours = ["#c9d4e3", "#9db4d0", "#6f93be", "#3f6fb0", "#1f3864"]  # light→dark
# --------------------------------------

x = np.arange(len(constructs))
n = len(results)
width = 0.8 / n

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (model, scores) in enumerate(results.items()):
    bars = ax.bar(x + (i - (n - 1) / 2) * width, scores, width,
                  label=model, color=colours[i % len(colours)],
                  edgecolor="white", linewidth=0.6)
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7)

ax.set_ylabel("Positive-class F1")
ax.set_xticks(x, constructs)
ax.set_ylim(0, 0.8)
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linewidth=0.5, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("rq1_model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()